# Assignment 2 — Create an Agent (Top-3 Web Sources)

This notebook implements an agent that:
- (Optionally) refines the user query
- calls `internet_search(query)` once
- fetches the **top 3** results with `fetch_url(url)`
- answers the question using only those sources, with citations


## 0) Setup

1. Create an account at https://openrouter.ai
2. Set your API key in your environment before running the agent:

**Mac/Linux**
```bash
export OPENROUTER_API_KEY="YOUR_KEY_HERE"
```

**Windows PowerShell**
```powershell
$env:OPENROUTER_API_KEY="YOUR_KEY_HERE"
```

Optional model override (defaults to free model):
```bash
export OPENROUTER_MODEL="nvidia/nemotron-3-nano-30b-a3b:free"
```

Install packages (if needed):
```bash
pip install langchain langchain-openai duckduckgo-search requests beautifulsoup4
```


## 1) System Prompt
This prompt forces the correct tool-use pattern (top-3 fetch + grounded answer).

In [ ]:
SYSTEM_PROMPT = """You are a web-research agent.

Goal: Answer the user's question using the content of the top-3 ranking websites.

Process (follow exactly):
1) Optional query refinement:
   - Rewrite the user's question into a better web search query.
   - Preserve the original intent.
2) Call internet_search(refined_query) exactly once.
3) From the returned ranked results, select the top 3 items.
   - Each item should have: title, url, snippet.
4) For each of the 3 URLs, call fetch_url(url) to retrieve page text.
5) Use ONLY the fetched page texts to answer.
   - If sources disagree, explain the conflict.
   - If information is missing from the top-3 pages, say what's missing.

Output requirements:
- Provide a clear, direct answer.
- Support key claims with citations as markdown links: [Title](URL)
- Prefer quoting or close paraphrasing from the fetched pages.
- Do NOT invent facts not present in the fetched pages.

Tooling notes:
- internet_search returns a JSON string containing a list of objects with keys: title, url, snippet.
"""


## 2) Tools: internet_search + fetch_url
This implements `internet_search` (DuckDuckGo) and `fetch_url` (requests + BeautifulSoup HTML-to-text).

If your course already provided `internet_search`, you can swap it in as long as the output keeps `title/url/snippet` in ranked order.

In [ ]:
import json
import re
from typing import Dict, List

import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
from langchain.tools import tool


def _clean_text(html: str, max_chars: int = 40_000) -> str:
    """Convert HTML to readable text and truncate for context safety."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text("\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    if len(text) > max_chars:
        text = text[:max_chars] + "\n\n[TRUNCATED]"
    return text


@tool
def internet_search(query: str) -> str:
    """Search the web and return JSON string of ranked results.

    Returns (JSON string):
      [ {"title": str, "url": str, "snippet": str}, ... ]

    Note: Uses DuckDuckGo via duckduckgo-search (no API key required).
    """
    results: List[Dict[str, str]] = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=10):
            url = r.get("href") or r.get("url")
            if not url:
                continue
            results.append(
                {
                    "title": (r.get("title") or "").strip(),
                    "url": url.strip(),
                    "snippet": (r.get("body") or r.get("snippet") or "").strip(),
                }
            )

    return json.dumps(results, ensure_ascii=False)


@tool
def fetch_url(url: str) -> str:
    """Fetch page content from a URL and return readable text (best-effort)."""
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; assignment-2-agent/1.0)",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()

    ctype = (resp.headers.get("Content-Type") or "").lower()
    if "text/html" in ctype:
        return _clean_text(resp.text)

    text = resp.text
    if len(text) > 40_000:
        text = text[:40_000] + "\n\n[TRUNCATED]"
    return text


## 3) Create the Agent (tool-using model via OpenRouter)
This uses LangChain's `create_tool_calling_agent` with OpenRouter (OpenAI-compatible endpoint).

In [ ]:
import os

from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


def _chat_model() -> ChatOpenAI:
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set.")

    model = os.environ.get("OPENROUTER_MODEL", "nvidia/nemotron-3-nano-30b-a3b:free")

    return ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        default_headers={
            "HTTP-Referer": os.environ.get("OPENROUTER_HTTP_REFERER", "http://localhost"),
            "X-Title": os.environ.get("OPENROUTER_X_TITLE", "assignment-2-agent"),
        },
        temperature=0.2,
    )


def create_agent() -> AgentExecutor:
    llm = _chat_model()
    tools = [internet_search, fetch_url]

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}"),
        ]
    )

    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True)


## 4) Run a question
Set `OPENROUTER_API_KEY` in your environment first, then run the cell below.

In [ ]:
executor = create_agent()
result = executor.invoke({"input": "What are the health benefits of green tea?"})
result["output"]
